In [273]:
import torch

In [274]:
def sample_sphere(n):
    """Sample a point on a sphere"""
    z = 2*torch.rand(n) - 1
    phi = 2*(torch.pi)*torch.rand(n)
    r = torch.sqrt(1-z**2)
    x, y = r*torch.cos(phi), r*torch.sin(phi)
    return torch.concat([x.reshape(-1,1),y.reshape(-1,1),z.reshape(-1,1)], axis=1)
    

In [275]:
coord = sample_sphere(4)

In [276]:
coord

tensor([[-0.1348,  0.5981,  0.7900],
        [ 0.4061, -0.9124, -0.0508],
        [-0.8187,  0.2694, -0.5071],
        [ 0.7644, -0.3316, -0.5529]])

In [277]:
torch.square(coord)

tensor([[0.0182, 0.3577, 0.6241],
        [0.1650, 0.8325, 0.0026],
        [0.6702, 0.0726, 0.2572],
        [0.5843, 0.1099, 0.3057]])

In [278]:
def check_validity(coord):
    assert  torch.all(torch.abs(torch.sum((torch.square(coord)), axis=1)  - 1) <= 1e-3)

In [279]:
check_validity(coord)

In [280]:
x,y = sample_sphere(4), sample_sphere(4)

In [281]:
x.shape, y.shape

(torch.Size([4, 3]), torch.Size([4, 3]))

In [282]:

def log_map(x,y):
    """Finds the minimum geodesic or minor arc for circle"""
    assert x.shape==y.shape
    n = x.shape[0]
    dot_prod = torch.tensor([torch.dot(x[i],y[i]) for i in range(n)]).reshape(-1,1)
    theta = torch.tensor([torch.arccos(dot) for dot in dot_prod]).reshape(-1,1)
    y_parallel = dot_prod * x
    y_perp = y - y_parallel
    y_perp_norm = torch.nn.functional.normalize(y_perp, 2, dim=-1)
    v = theta * y_perp_norm
    return v


        

In [283]:
v = log_map(x,y)

In [284]:
v.shape

torch.Size([4, 3])

In [285]:
def test_tangentness(x, v, tol=1e-5):
    assert x.shape==v.shape
    n = x.shape[0]
    dot_prod = torch.tensor([torch.dot(x[i],v[i]) for i in range(n)])
    tol_check = torch.abs(dot_prod)<=tol
    assert torch.all(tol_check)


In [286]:
test_tangentness(x,v)

In [287]:
def log_map_length_test(x,y,v, tol=1e-5):
    assert x.shape == y.shape
    assert x.shape == v.shape
    n = x.shape[0]
    theta = torch.tensor([torch.arccos(torch.dot(x[i], y[i])) for i in range(n)])
    v_norm = torch.linalg.vector_norm(v, ord=2, dim=-1)
    assert torch.all(torch.abs(v_norm - theta) <= tol)


In [288]:
log_map_length_test(x,y,v)

In [289]:
def exp_map(x,v):
    assert x.shape==v.shape
    n = x.shape[0]
    v_norm = torch.nn.functional.normalize(v, p=2, dim=-1)
    theta = torch.linalg.vector_norm(v, ord=2, dim=-1)
    y = torch.cos(theta).reshape(-1,1) * x + torch.sin(theta).reshape(-1,1) * v_norm
    return y
    

In [290]:
exp_map(x,v)

tensor([[ 0.6207,  0.0573,  0.7819],
        [-0.4931, -0.6067,  0.6235],
        [ 0.7806, -0.3230, -0.5350],
        [ 0.2587, -0.9379,  0.2311]])

In [291]:
def test_exp(x,y, tol=1e-5):
    assert torch.all(torch.abs(exp_map(x, log_map(x,y)) - y) <= tol)

In [292]:
test_exp(x,y)

In [293]:
def test_unit_norm(x,v, tol=1e-5):
    assert torch.all(torch.abs(torch.linalg.vector_norm(exp_map(x,v), ord=2, dim=-1)-1) <= tol)

In [294]:
test_unit_norm(x,v)

In [295]:
def premetric_d(x,y):
    assert x.shape == y.shape
    n = x.shape[0]
    return torch.tensor(
        [torch.arccos(torch.clamp(torch.dot(x[i], y[i]), min=-1, max=1)) for i in range(n)]
        )

In [296]:
premetric_d(x,y)

tensor([2.4661, 2.0878, 1.2349, 2.6988])

In [297]:
def test_d_x_x_zero(x,tol=1e-3):
    assert torch.all(
        torch.abs(
            premetric_d(x,x) - 0
        ) <= tol
    )

In [298]:
test_d_x_x_zero(x)

In [299]:
def test_d_symmetry(x,y,tol=1e-3):
    assert torch.all(
        torch.abs(
            premetric_d(x,y) - premetric_d(y,x)
        ) <= tol
    )

In [300]:
test_d_symmetry(x,y)

In [301]:
def test_d_non_neg(x,y):
    assert torch.all(
        premetric_d(x,y) >= 0
    )

In [302]:
test_d_non_neg(x,y)

In [303]:
def grad_d(x, y):
    v = log_map(x,y)
    v_norm = torch.linalg.vector_norm(v, ord=2, dim=-1).reshape(-1,1)
    return -v * torch.reciprocal(v_norm)

In [304]:
g = grad_d(x,y)

In [305]:
test_tangentness(x,g)

In [306]:
test_unit_norm(x,g)

In [307]:
def test_against_log_map(x,y, tol=1e-3):
    assert torch.all(torch.abs(
        torch.nn.functional.normalize(log_map(x,y),p=2,dim=-1) + grad_d(x,y)
    ) <= tol
    )

In [308]:
test_against_log_map(x,y)

In [309]:
def get_time_sched_and_derivative(t):
    assert torch.all(t < 1)
    assert torch.all(t>=0)
    return 1-t, -1*torch.reciprocal(1-t)

In [310]:
get_time_sched_and_derivative(torch.tensor([0,0.5]))

(tensor([1.0000, 0.5000]), tensor([-1., -2.]))

In [311]:
def conditional_vf(x, x1, t):
    assert x.shape == x1.shape
    assert x.shape[0] == t.shape[0]
    grad = grad_d(x,x1)
    pre = premetric_d(x,x1).reshape(-1,1)
    _, log_deriv = get_time_sched_and_derivative(t)
    log_deriv = log_deriv.reshape(-1,1)
    return log_deriv * pre * grad * \
            torch.reciprocal(
                torch.square(
                torch.linalg.vector_norm(grad,ord=2,dim=-1)
                ).reshape(-1,1)
                )

    


In [312]:
def get_time_samples(n):
    return torch.rand(n)

In [313]:
t = get_time_samples(4)

In [314]:
cvf = conditional_vf(x,y,t)